# Market Data Retrieval via SLUG or Market ID

In [1]:
import time
import requests
import pandas as pd

In [2]:
GAMMA = "https://gamma-api.polymarket.com"
DATA  = "https://data-api.polymarket.com"

def get_market(market_id: str = None, slug: str = None) -> dict:
    if (market_id is None) == (slug is None):
        raise ValueError("Provide either a market_id or slug.")

    if market_id:
        r = requests.get(f"{GAMMA}/markets/{market_id}", timeout=30)
    else:
        r = requests.get(f"{GAMMA}/markets/slug/{slug}", timeout=30)

    r.raise_for_status()
    return r.json()


def fetch_all_trades(condition_id: str, limit: int = 10000):
    all_trades = []
    offset = 0

    while True:
        params = {
            "market": condition_id,
            "limit": limit,
            "offset": offset,
            "takerOnly": "true"
        }

        r = requests.get(f"{DATA}/trades", params=params, timeout=30)
        r.raise_for_status()
        trades = r.json()

        if not trades:
            break

        all_trades.extend(trades)
        offset += len(trades) # ensure we get all trades in limit size intervals
        time.sleep(0.05)

    return all_trades


def download_market_trades(market_id: str = None,
                           slug: str = None,
                           out_prefix: str = "polymarket"):

    market = get_market(market_id=market_id, slug=slug)
    condition_id = market["conditionId"]

    trades = fetch_all_trades(condition_id)
    trades_df = pd.DataFrame(trades)

    filename = f"market_csvs/{out_prefix}_trades.csv"
    trades_df.to_csv(filename, index=False)

    print(f"Saved {len(trades_df)} trades to {filename}")

    return trades_df

In [3]:
market = download_market_trades(slug="trumps-travel-ban-removed-by-january-31", out_prefix="trumps-31")

Saved 150 trades to market_csvs/trumps-31_trades.csv


In [4]:
market_test = pd.read_csv("market_csvs/trumps-31_trades.csv")
market_test.drop(columns = ["name", "pseudonym", "bio", "profileImage", "profileImageOptimized", "transactionHash"])
market_test["size"].sum()


np.float64(10141.547676)

This works! market volume is the same as the sum of the volume, our retrieving works well and extra columns are dropped!